# Platform Setup

Creates Unity Catalog schemas and external volumes backed by Amazon S3.

This notebook requires elevated platform permissions and is not part of the
scheduled ingestion workflow.

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("platform_schema", "sbx_platform")
dbutils.widgets.text("bucket_name", "")

In [0]:

catalog = dbutils.widgets.get("catalog").strip()
platform_schema = dbutils.widgets.get("platform_schema").strip()
bucket_name = dbutils.widgets.get("bucket_name").strip()

if not catalog:
    raise ValueError("catalog is required.")

if not platform_schema:
    raise ValueError("platform_schema is required.")

if not bucket_name:
    raise ValueError(
        "bucket_name is required. Enter the S3 bucket name in the widget."
    )

In [0]:
schemas = [
    platform_schema,
    "sbx_bronze",
    "sbx_silver",
    "sbx_gold",
    "sbx_ops",
]

for schema_name in schemas:
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema_name}`"
    )

print(f"Created or verified {len(schemas)} schemas.")

![image_1791639164095.png](./image_1791639164095.png "image_1791639164095.png")

In [0]:
volume_locations = {
    "landing": f"s3://{bucket_name}/landing",
    "quarantine": f"s3://{bucket_name}/quarantine",
    "checkpoints": f"s3://{bucket_name}/checkpoints",
}

for volume_name, location in volume_locations.items():
    spark.sql(
        f"""
        CREATE EXTERNAL VOLUME IF NOT EXISTS
        `{catalog}`.`{platform_schema}`.`{volume_name}`
        LOCATION '{location}'
        """
    )

print(f"Created or verified {len(volume_locations)} external volumes.")


In [0]:
volume_root = f"/Volumes/{catalog}/{platform_schema}"

landing_root = f"{volume_root}/landing"
quarantine_root = f"{volume_root}/quarantine"
checkpoint_root = f"{volume_root}/checkpoints"

transactions_path = f"{landing_root}/transactions"
routes_path = f"{landing_root}/routes"

In [0]:
for volume_name in volume_locations:
    volume_path = f"{volume_root}/{volume_name}"
    files = dbutils.fs.ls(volume_path)

    print(f"Volume accessible: {volume_path}")

In [0]:
transactions_files = dbutils.fs.ls(transactions_path)
routes_files = dbutils.fs.ls(routes_path)

if not transactions_files:
    raise ValueError(
        f"No transaction source files found at {transactions_path}"
    )

if not routes_files:
    raise ValueError(
        f"No route source files found at {routes_path}"
    )

display(transactions_files)
display(routes_files)

In [0]:
# print("Platform setup completed successfully.")
# print(f"Catalog: {catalog}")
# print(f"Platform schema: {platform_schema}")
# print(f"Landing volume: {landing_root}")
# print(f"Quarantine volume: {quarantine_root}")
# print(f"Checkpoint volume: {checkpoint_root}")